<a href="https://colab.research.google.com/github/updalla-apshir/machine-learning-projects/blob/main/Project_4_Fake_News_Prediction_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import re

# NLTK needs a file containing English stopwords install the language datasets  

In [ ]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
news_data = fetch_openml(name='Fake-News')

In [ ]:
news_data.data.head()

,Unnamed:_0,title,text,label
0,8476,You Can Smell Hillarys Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"Kaydee King (KaydeeKing) November 9, 2016 The...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


In [ ]:
# checking misssing values
print(news_data.data.isnull().sum())

#Check if the whole dataset has any null
print(news_data.data.isnull().values.any())

Unnamed:_0    0
title         0
text          0
label         0
dtype: int64
False


In [ ]:
# checking imbalanced data
news_data.data['label'].value_counts()

,count
label,
REAL,3171
FAKE,3164


In [ ]:
# merging the title and text
news_data.data['content'] = news_data.data['title'] + '-' + news_data.data['text']

In [ ]:
news_data.data

,Unnamed:_0,title,text,label,content
0,8476,You Can Smell Hillarys Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE,"You Can Smell Hillarys Fear-Daniel Greenfield,..."
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE,Watch The Exact Moment Paul Ryan Committed Pol...
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL,Kerry to go to Paris in gesture of sympathy-U....
3,10142,Bernie supporters on Twitter erupt in anger ag...,"Kaydee King (KaydeeKing) November 9, 2016 The...",FAKE,Bernie supporters on Twitter erupt in anger ag...
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL,The Battle of New York: Why This Primary Matte...
...,...,...,...,...,...
6330,4490,State Department says it can't find emails fro...,The State Department told the Republican Natio...,REAL,State Department says it can't find emails fro...
6331,8062,The P in PBS Should Stand for Plutocratic or P...,The P in PBS Should Stand for Plutocratic or P...,FAKE,The P in PBS Should Stand for Plutocratic or P...
6332,8622,Anti-Trump Protesters Are Tools of the Oligarc...,Anti-Trump Protesters Are Tools of the Oligar...,FAKE,Anti-Trump Protesters Are Tools of the Oligarc...
6333,4021,"In Ethiopia, Obama seeks progress on peace, se...","ADDIS ABABA, Ethiopia President Obama convened...",REAL,"In Ethiopia, Obama seeks progress on peace, se..."


## Stemming

Stemming is the process or reducing a word to its root word

Example:
playing,played,plays --------> play

In [ ]:
port_stem = PorterStemmer()

In [ ]:
def port_steam(content):
  port_steamed = re.sub('[^a-zA-Z]',' ', content)
  port_steamed = port_steamed.lower()
  port_steamed = port_steamed.split()
  port_steamed = [port_stem.stem(word) for word in port_steamed if not word in stopwords.words('english')]
  port_steamed = ' '.join(port_steamed)
  return port_steamed



In [ ]:
news_data.data['content'] = news_data.data['content'].apply(port_steam)

In [ ]:
print(news_data.data['content'])

0       You Can Smell Hillarys Fear-Daniel Greenfield,...
1       Watch The Exact Moment Paul Ryan Committed Pol...
2       Kerry to go to Paris in gesture of sympathy-U....
3       Bernie supporters on Twitter erupt in anger ag...
4       The Battle of New York: Why This Primary Matte...
                              ...                        
6330    State Department says it can't find emails fro...
6331    The P in PBS Should Stand for Plutocratic or P...
6332    Anti-Trump Protesters Are Tools of the Oligarc...
6333    In Ethiopia, Obama seeks progress on peace, se...
6334    Jeb Bush Is Suddenly Attacking Trump. Here's W...
Name: content, Length: 6335, dtype: object


In [ ]:
news_data.data["label"] = news_data.data["label"].map({
    "FAKE": 0,
    "REAL": 1
})

In [ ]:
# Separating the the data and label
X = news_data.data['content'].values
Y = news_data.data['label'].values

In [ ]:
Y.shape

(6335,)

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(X,Y, test_size=0.2, stratify=Y, random_state=2)


## Create vectorizer
changing text into numerical values

In [ ]:
vectorizer = TfidfVectorizer()
vectorizer.fit(x_train)

TfidfVectorizer()

In [ ]:
x_train = vectorizer.transform(x_train)
x_test = vectorizer.transform(x_test)

Training Model

In [ ]:
model = LogisticRegression()

In [ ]:
# feed model with training data
model.fit(x_train,y_train)

LogisticRegression()

In [ ]:
# predict the training data
predict_training = model.predict(x_train)

In [ ]:
# evaluate the predicted data
training_accuracy = accuracy_score(predict_training,y_train)

In [ ]:
print('training accuracy: ', training_accuracy)

training accuracy:  0.9717837411207577


In [ ]:
# predict the testing data
predict_testing = model.predict(x_test)

In [ ]:
# evaluate the testing accuracy
testing_accuracy = accuracy_score(predict_testing,y_test)

In [ ]:
print('testing accuracy: ', testing_accuracy)

testing accuracy:  0.9463299131807419


## build prediction system

In [ ]:
input = ('Government Announces New Education Program he government announced a new education program aimed at improving digital skills among students. The program will provide technology resources and training for schools across the country. Officials said the project will begin next year after completing the required planning process.  ')

stem_input = port_steam(input)

input_vectorizer= vectorizer.transform([stem_input])

prediction = model.predict(input_vectorizer)

if prediction[0] == 0:
  print('Fake News')
else:
  print('Real News')

Real News


In [ ]:
print(news_data.data['content'].iloc[0])

smell hillari fear daniel greenfield shillman journal fellow freedom center new york writer focus radic islam nin final stretch elect hillari rodham clinton gone war fbi nthe word unpreced thrown around often elect ought retir still unpreced nomine major polit parti go war fbi nbut that exactli hillari peopl done coma patient wake watch hour cnn hospit bed would assum fbi director jame comey hillari oppon elect nthe fbi attack everyon obama cnn hillari peopl circul letter attack comey current media hit piec lambast target trump wouldnt surpris clinton alli start run attack ad fbi nthe fbi leadership warn entir left wing establish form lynch mob continu go hillari fbi credibl attack media democrat preemptiv head result investig clinton foundat hillari clinton nthe covert struggl fbi agent obama doj peopl gone explos public nthe new york time compar comey j edgar hoover bizarr headlin jame comey role recal hoover fbi fairli practic admit front spout nonsens boston globe publish column ca

In [ ]:
print(news_data.data['label'].iloc[3])

0
